# 拓扑码的神经解码器——独立复现

本 Notebook 独立复现 G. Torlai 和 R. G. Melko 的论文 *Neural Decoder for Topological Codes*，发表于 **Physical Review Letters 119**，030501（2017），[doi:10.1103/PhysRevLett.119.030501](https://doi.org/10.1103/PhysRevLett.119.030501)。论文 PDF 保存在 `paper/docs/`。

**职责分工。** 本 Notebook 是实验的程序入口，负责配置、阶段顺序、逐测试样本记录和结果展示。`ai_qec/` 提供共享的数据生成、epoch/minibatch 训练和 Gibbs 解码，并负责 Toric code 与噪声模型、精确 MWPM、数据集校验、指标和报告构建，以及 run 记录。

| 论文算法 1 的步骤 | Notebook 章节 |
|---|---|
| 第 1–2 行：`e₀`、`S₀ = S(e₀)` | 第 3 节：数据生成 |
| 第 3 行：`RBM = {e, S=S₀, h}`（通过 CD-k 训练） | 第 4 节：训练 |
| 第 4–8 行：钳制 syndrome，执行 Gibbs 采样直到 `S(e) = S₀`，令 `r = e` | 第 5 节：神经解码 |
| 根据 `e₀ ⊕ r` 判断逻辑失败 | 第 6 节：基准对照 |

下方单元格内的参数只用于小规模冒烟验证。其输出可以验证实验路径，但不能视为论文最终图表的数值复现，也不足以支持阈值结论。

**冒烟输出的预期形态。** 这组参数（256 个训练样本、8 个 epoch，共 32 次梯度更新）训练量刻意不足，测试样本通常全部超时，于是 `P_fail = 1.0`、同调计数全为零。这是训练不足的预期表现，不是程序错误。本 Notebook 的冒烟只验证阶段顺序与产物契约；解码成功路径的正确性由 `tests/` 覆盖（`python3 -m unittest discover -s tests -t .`）。

In [ ]:
# 导入共享 setup，并由已安装的 ai_qec 包定位项目源码根目录。
from __future__ import annotations

from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np

import ai_qec.notebook_api as qec

PROJECT_ROOT = qec.find_project_root()


## 1. 实验配置契约

本节定义 `EXPERIMENT_CONFIG`：码、噪声、数据、模型及训练参数。`qec.build_experiment(...)` 检查它选择了本 Notebook 支持的数据生成器和模型，并构建 `code` 与 `noise`——这是仅由实验参数决定的一对对象。下一节启动 run 时会用同一份参数重建这两个对象并比对，以防配置单元格改过却没有重新执行；其余参数都从合并后的配置读取，不会过期。本 Notebook 不读取 `configs/` 下的 YAML。脚本 runner 的 `flow` 在此由 Notebook 的阶段顺序取代。

需要检查并行链时，可在 `EXPERIMENT_CONFIG` 中将 `training.decoder.parallel_chains` 从 `1` 改为 `64`，再开始一次新的 run；该结果应单独标为并行链实验。

若只想用更大的 Gibbs 步数预算等解码设置重新评估已训练的模型，把 `REUSE_CHECKPOINT_FROM_RUN` 设为源 run 的目录名，并让 `EXPERIMENT_CONFIG` 与源 run 保持一致（只改 `training.decoder` 或 `training.device`）。此时第 3 节校验并引用源 run 的数据集，第 4 节复用其 `best.pt`，不重新训练。

In [ ]:
# 定义码、噪声、数据、模型和训练参数。
EXPERIMENT_CONFIG = {
    'qec': {'code': 'toric_code', 'distance': 4, 'rounds': 1, 'task': 'code_capacity'},
    'noise': {'model': 'phase_flip', 'p_error': 0.08},
    'data': {
        'generator': 'toric_code_capacity',
        'train_samples': 256, 'validation_samples': 64, 'test_samples': 24,
        'dataset_id': 'torlai_melko_2017_l4_p008_smoke',
        'output_dir': 'datasets', 'batch_size': 64,
        'preprocessing': {'representation': 'error_syndrome'},
    },
    'model': {'implementation': 'joint_error_syndrome_rbm', 'hidden_units': 16, 'init_width': 0.01},
    'training': {
        'trainer': 'rbm_cd', 'device': 'cuda', 'epochs': 8, 'batch_size': 64,
        'learning_rate': 0.05, 'cd_steps': 1, 'weight_decay': 0.0001,
        'decoder': {'burn_in': 25, 'max_steps': 400, 'parallel_chains': 1},
    },
}
# 复用已训练模型：填入 runs/ 下源 run 的目录名，即跳过第 4 节训练，只执行第 5–6 节；None 表示完整训练。
# 此时 EXPERIMENT_CONFIG 必须与源 run 一致，只允许 training.decoder 与 training.device 不同。
REUSE_CHECKPOINT_FROM_RUN = None
code, noise = qec.build_experiment(
    EXPERIMENT_CONFIG, generator='toric_code_capacity', model='joint_error_syndrome_rbm',
)

## 2. 校验配置并创建独立 run

本节定义 `RUN_CONFIG`：schema、执行环境、实验标识、研究方向、可复现设置和输出位置。`qec.start_notebook_run(...)` 先重建并比对第 1 节构建的 `code` 与 `noise`，合并并严格校验两组配置，检查内核与设备，展示完整有效配置；全部通过后才创建唯一 run，并将有效配置保存为 `config.yaml`。后续阶段通过 `run.stage(...)` 记录状态、耗时及产物哈希。

In [ ]:
# 校验配置并创建本次执行的唯一 run。
PAPER_PATH = PROJECT_ROOT / 'paper' / 'docs' / 'A Neural Decoder for Topological Codes.pdf'
assert PAPER_PATH.is_file(), PAPER_PATH

RUN_CONFIG = {
    'schema_version': 1,
    'execution': {'conda_env': 'quantum'},
    'experiment': {
        'name': 'torlai_melko_2017_rbm_smoke',
        'title': 'Joint RBM decoder for a toric-code capacity experiment',
        'description': 'Small-lattice smoke of the Torlai--Melko (2017) joint RBM workflow.',
        'owner': 'researcher',
        'tags': ['ai-qec', 'toric-code', 'rbm', 'paper-reproduction', 'smoke'],
    },
    'topic': {'direction': 'D1', 'subtopic': 'D1.2', 'name': 'generative_neural_decoding'},
    'reproducibility': {
        'seeds': [17], 'deterministic': True, 'save_environment': True,
        'save_git_commit': True, 'require_clean_worktree': False,
    },
    'outputs': {'runs_root': 'runs'},
}

run, DEVICE, SEED = qec.start_notebook_run(
    RUN_CONFIG, EXPERIMENT_CONFIG, project_root=PROJECT_ROOT,
    notebook='paper/srcs/torlai_melko_2017.ipynb',
    code=code, noise=noise,
)
config, RUN_DIR = run.config, run.run_dir
RUN_DIR

## 3. 数据生成——论文算法 1 第 1–2 行

共享数据生成器对每个 split 独立采样相位翻转错误链 `e₀`，并计算完美测量条件下的顶点 syndrome `S₀ = S(e₀)`。数据集按内容身份寻址且不可覆盖：如果对应生成规范的数据目录已经存在，则先校验再复用；否则将各 split 写入临时目录，只有全部写入成功后才提交为正式数据集。

In [ ]:
# 论文算法 1 第 1–2 行：共享生成器逐 split、逐 batch 采样 e0 并计算 S0。
# resolve_dataset 负责引用源 run、复用已有数据或生成新数据及其 manifest。
DATASET_DIR = qec.data_output_dir(config, PROJECT_ROOT)
SOURCE_RUN = None if REUSE_CHECKPOINT_FROM_RUN is None else qec.load_source_run(
    REUSE_CHECKPOINT_FROM_RUN, runs_root=qec.resolve_project_path(PROJECT_ROOT, config['outputs']['runs_root']))

with run.stage('generate_data') as step:
    DATASET_DIR, dataset_manifest = qec.resolve_dataset(
        run, config, PROJECT_ROOT, step=step, source_run=SOURCE_RUN)

splits = {name: qec.load_toric_split(DATASET_DIR, name) for name in ('train', 'validation', 'test')}
{
    'dataset_dir': str(DATASET_DIR.relative_to(PROJECT_ROOT)),
    'reused': step.get('reused_immutable', False),
    'sample_counts': dataset_manifest['sample_counts'],
    'mean_error_weight': {name: float(split.physical_error.sum(axis=1).mean()) for name, split in splits.items()},
}


## 4. 训练——论文算法 1 第 3 行的联合 RBM

RBM 建模可见单元 `[e | S]` 的联合分布。共享 PyTorch 训练器每个 epoch 打乱训练集，并对每个 minibatch 执行一次 CD-k 更新。验证集一步重建 BCE 最低的 checkpoint 保存为 `best.pt`；这一选择规则是本项目的实验约定，并非来自原论文。

设置了 `REUSE_CHECKPOINT_FROM_RUN` 时，本节不训练：先核对源 run 记录的 `best.pt` 哈希及其训练数据集，再拷入本次 run，并沿用源 run 的训练记录绘制曲线。第 5 节解码后，会在两次 run 共同的步数预算内逐条核对恢复链，不一致即判定本次 run 失败。

In [ ]:
# 论文算法 1 第 3 行：共享训练器负责 DataLoader、CD-k 更新、验证和 best/last checkpoint。
BEST_CHECKPOINT = RUN_DIR / 'checkpoints' / 'best.pt'

if SOURCE_RUN is not None:
    with run.stage('reuse_checkpoint') as step:
        training_summary, step['outputs'] = qec.reuse_training_outputs(
            SOURCE_RUN, run_dir=RUN_DIR, config=config, dataset_manifest=dataset_manifest)
        step['source_run'] = SOURCE_RUN.run_id
    print(f"reused checkpoint from {SOURCE_RUN.run_id}; training skipped")
else:
    with run.stage('train') as step:
        training_result = qec.run_rbm_training(config, PROJECT_ROOT, RUN_DIR)
        training_summary = training_result.summary
        step['outputs'].extend(training_result.artifacts)

history = training_summary['history']
best_epoch = int(training_summary['metrics']['selected_epoch'])
training_summary['metrics']


In [ ]:
# 绘制训练与验证集的一步重建 BCE 曲线，并标出被选中的 epoch。
fig, ax = plt.subplots(figsize=(6, 3.5))
epochs = [row['epoch'] for row in history]
ax.plot(epochs, [row['train_reconstruction_bce'] for row in history], marker='o', label='Train')
ax.plot(epochs, [row['validation_reconstruction_bce'] for row in history], marker='o', label='Validation')
ax.axvline(best_epoch, color='grey', linestyle='--', linewidth=1, label=f'Selected epoch {best_epoch}')
ax.set_xlabel('epoch')
ax.set_ylabel('One-step reconstruction BCE')
ax.legend()
ax.grid(alpha=.25)
plt.show()

## 5. 神经解码——论文算法 1 第 3–8 行

将 syndrome 单元钳制为测得的 `S₀`，交替执行块 Gibbs 更新 `h ~ p(h | e, S₀)` 与 `e ~ p(e | h)`，直到采样得到满足 `S(e) = S₀` 的错误链，并以该链作为恢复链 `r`。论文正文也说明了固定采样步数截止和失败计数；这里用 `burn_in` 与 `max_steps` 明确配置预算：仅在预热后检查兼容性，超出预算则记为超时和逻辑失败。当前算法选取第一条兼容链，并不比较各同调类的概率；`parallel_chains > 1` 是另行标注的扩展实验。

In [ ]:
# 已校验的配置由共享 RBMGibbsDecoder 在评估阶段读取。
config['training']['decoder']


In [ ]:
# 在 test split 上逐样本调用共享 Gibbs 解码器；超时记为逻辑失败（见 DecodingRecord）。
PREDICTIONS_PATH = RUN_DIR / 'predictions' / 'toric_rbm_eval.npz'
METRICS_PATH = RUN_DIR / 'metrics.json'

with run.stage('evaluate') as step:
    rbm, _ = qec.load_model(config, str(BEST_CHECKPOINT))
    decoder_config = config['training']['decoder']
    decoder = qec.RBMGibbsDecoder(
        rbm, code, burn_in=int(decoder_config['burn_in']),
        max_steps=int(decoder_config['max_steps']),
        parallel_chains=int(decoder_config.get('parallel_chains', 1)), device=DEVICE,
    )
    test = splits['test']
    record = qec.DecodingRecord(len(test.physical_error), code.num_data_qubits)

    for index, (e0, s0) in enumerate(zip(test.physical_error, test.syndrome, strict=True)):
        started = time.perf_counter_ns()
        result = decoder.decode(s0, rng=qec.decoding_rng(config, index))
        latency_ms = (time.perf_counter_ns() - started) / 1e6
        recovery = result.recovery
        failed = None if recovery is None else bool(code.logical_failure(e0[None, :], recovery[None, :])[0])
        record.add(index, recovery=recovery, steps=result.steps, latency_ms=latency_ms, failed=failed)

    metrics = record.metrics('test')
    record.save(PREDICTIONS_PATH, dataset=test, parallel_chains=decoder.parallel_chains, device=DEVICE)
    qec.write_json(METRICS_PATH, {'metrics': metrics})
    step['outputs'] += [PREDICTIONS_PATH, METRICS_PATH]
    if SOURCE_RUN is not None:
        qec.verify_source_consistency(
            SOURCE_RUN, RUN_DIR, step=step, config=config, syndromes=test.syndrome,
            recovery_valid=record.recovery_valid, gibbs_steps=record.gibbs_steps, recovery=record.recovery)

metrics


## 6. 与精确 MWPM 的基准对照

在同一个 test split 上使用无外部依赖的精确 MWPM 参照解码器。通过 `error XOR recovery` 的同调判断逻辑失败，并用 Wilson 95% 置信区间比较失败率。精确匹配器有缺陷数上限；超过上限的 syndrome 改用 PyMatching，它同样求最小权重完美匹配，两者只在多个等权最优解之间的取舍上可能不同。报告中的 `pymatching_fallback_shots` 记录改用的条数。

In [ ]:
# MWPM 基线逐样本解码，与 RBM 结果汇总为 benchmark 报告，并结束本次 run。
with run.stage('benchmark') as step:
    # 精确 MWPM；缺陷数超过其上限的 syndrome 改用 PyMatching（同样是最小权重完美匹配），并记录条数
    mwpm_recoveries, max_exact_defects, mwpm_fallback_shots = qec.mwpm_reference_recoveries(code, test.syndrome)
    report, benchmark_metrics = qec.build_toric_benchmark_report(
        code, split='test', p_error=noise.p_error, errors=test.physical_error, rbm_recoveries=record.recovery,
        rbm_valid=record.recovery_valid, rbm_failures=record.logical_failure, decoder_latency_ms=record.decoder_latency_ms,
        parallel_chains=decoder.parallel_chains, device=DEVICE, mwpm_recoveries=mwpm_recoveries,
        max_exact_defects=max_exact_defects, mwpm_fallback_shots=mwpm_fallback_shots)
    step['outputs'] += qec.write_benchmark_outputs(RUN_DIR, report, benchmark_metrics)

run.finish()
print(f'run complete: {RUN_DIR}')
report

In [ ]:
# 绘制 RBM Gibbs 解码器与精确 MWPM 的逻辑失败率，误差条为 Wilson 95% 置信区间。
labels = ['RBM Gibbs', 'Exact MWPM']
rates = [report['rbm']['logical_error_rate'], report['mwpm_exact']['logical_error_rate']]
intervals = [report['rbm']['wilson_95'], report['mwpm_exact']['wilson_95']]
yerr = np.array([[rate - interval[0] for rate, interval in zip(rates, intervals)],
                 [interval[1] - rate for rate, interval in zip(rates, intervals)]])
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(labels, rates, yerr=yerr, capsize=5, color=['#4C78A8', '#F58518'])
ax.set_ylim(0, 1.05)
ax.set_ylabel('Logical failure rate')
ax.set_title(f"{code.name} d={report['lattice_size']}, p={report['p_error']}: smoke run comparison")
ax.grid(axis='y', alpha=.25)
plt.show()

In [ ]:
# 并排展示 RBM 与精确 MWPM 的逻辑类分布，便于观察逻辑失败类型。
# 标签由码的逻辑位数决定：toric 有 2 个逻辑位（四个同调扇区），k=1 的码只有 '0'/'1'。
sectors = qec.logical_class_labels(code)
x = np.arange(len(sectors))
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(x - .18, [report['rbm']['homology_counts'][s] for s in sectors], .36, label='RBM accepted recovery', color='#4C78A8')
ax.bar(x + .18, [report['mwpm_exact']['homology_counts'][s] for s in sectors], .36, label='Exact MWPM', color='#F58518')
ax.set_xticks(x, sectors)
ax.set_xlabel('Logical class of physical error XOR recovery')
ax.set_ylabel('Test samples')
ax.legend()
ax.set_title('Logical classes of the closed cycle')
plt.show()

## 本次 run 能说明什么

一次 Notebook 执行会生成或复用 code-capacity 数据，以 CD-k 训练联合 RBM，执行 syndrome 钳制的 Gibbs 解码，通过 `error XOR recovery` 的同调判断逻辑失败，并在同一 test split 上与精确 MWPM 对照。各阶段均记录在 `run_manifest.json` 中。冒烟配置刻意保持小规模，不能支持科学性能结论。论文规模的结论还需要预先确定格点大小、错误率和随机种子网格，采集足够多的样本，并记录 MWPM 对照中改用 PyMatching 的条数。